# QSVM estilo Farooq — faixas AQI (t+24h)

Notebook **independente** (legado adaptado para a Névoa / artigo Farooq):

- Features Farooq (`pm25/temperature` × min/max/median/variance)
- Alvos em **t+24h** por faixa EPA e extremo:
  1. `aqi_good_vs_moderate` — Good vs Moderate (mais perto do artigo)
  2. `aqi_bad` — pior que Moderate (Unhealthy_Sensitive+)
  3. `extreme_p90` — extremo P90 (alerta)
- Clássicos (8D) + QSVM (PCA→2 + MinMax + ZZFeatureMap)
- Multi-seed + painel por tarefa

**Kaggle:** Internet **ON** · Accelerator **None** (CPU)

> O benchmark metodológico justo de extremos continua em `05_farooq_fair_benchmark.ipynb`.


## 1. Instalação


In [ ]:
%pip install -q qiskit qiskit-machine-learning qiskit-aer scikit-learn pandas numpy matplotlib seaborn joblib scipy


## 2. Configuração

| Modo | Sample | Seeds | Tempo típico (3 tarefas × QSVM) |
|---|---:|---:|---|
| `small` | 80/40/40 | 1 | ~5–15 min |
| `medium` | 200/60/80 | 5 | ~1–3 h |
| `large` | 500/150/200 | 10 | várias horas |


In [ ]:
from pathlib import Path
import math

ON_KAGGLE = Path('/kaggle/working').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('artifacts/kaggle_farooq_aqi_nb')
WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / 'data'
PLOTS = WORK / 'plots'
DATA.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)

# ==========================================================
# MODO: 'small' | 'medium' | 'large'
# ==========================================================
MODE = 'small'

PRESETS = {
    'small': dict(train=80, val=40, test=40, seeds=[42], q_variants='farooq_only'),
    'medium': dict(train=200, val=60, test=80, seeds=[42, 7, 11, 13, 21], q_variants='both'),
    'large': dict(
        train=500, val=150, test=200,
        seeds=[42, 7, 11, 13, 21, 29, 37, 41, 47, 53],
        q_variants='both',
    ),
}

cfg_mode = PRESETS[MODE]
TRAIN_SIZE = cfg_mode['train']
VAL_SIZE = cfg_mode['val']
TEST_SIZE = cfg_mode['test']
SEEDS = list(cfg_mode['seeds'])
PRIMARY_SEED = SEEDS[0]

STATION = 'Aotizhongxin'
HORIZON_HOURS = 24
EXTREME_PERCENTILE = 0.90
ROLL_WINDOW = 24
ROLL_MIN_PERIODS = 12
PCA_QUBITS = 2

# Tarefas alinhadas à Névoa / Farooq
TASKS = [
    ('aqi_good_vs_moderate', 'aqi_good_vs_moderate', 'Good vs Moderate (EPA)'),
    ('aqi_bad', 'aqi_bad', 'Pior que Moderate (aqi_bad)'),
    ('extreme_p90', 'extreme_p90', 'Extremo P90'),
]

ALL_Q_VARIANTS = [
    ('Q_farooq_pipeline', 'minmax_0_1', 1),
    ('Q_farooq_0pi', 'minmax_0_pi', 2),
]
Q_VARIANTS = [ALL_Q_VARIANTS[0]] if cfg_mode['q_variants'] == 'farooq_only' else ALL_Q_VARIANTS


def estimate_minutes(n_train, n_test, n_seeds, n_tasks, variants):
    classical = 0.08 * n_seeds * n_tasks
    q = 0.0
    for _, _, reps in variants:
        pairs = n_train * n_train + n_test * n_train
        sec = pairs * 0.0023 * (1.0 + 0.1 * (reps - 1))
        q += sec * n_seeds * n_tasks / 60.0
    return classical + q + 2.0


est = estimate_minutes(TRAIN_SIZE, TEST_SIZE, len(SEEDS), len(TASKS), Q_VARIANTS)
print('=' * 60)
print(f'MODE={MODE}')
print(f'sample train/val/test = {TRAIN_SIZE}/{VAL_SIZE}/{TEST_SIZE}')
print(f'seeds ({len(SEEDS)}): {SEEDS}')
print(f'tasks: {[t[0] for t in TASKS]}')
print(f'QSVM variants: {[v[0] for v in Q_VARIANTS]}')
print(f'Estimativa: ~{est:.0f} min (faixa {est*0.7:.0f}–{est*1.5:.0f})')
print('WORK=', WORK)
print('=' * 60)


## 3. Download dos dados (UCI Beijing)


In [ ]:
import zipfile
from io import BytesIO
from urllib.request import urlopen
import pandas as pd

UCI_URLS = [
    'https://archive.ics.uci.edu/ml/machine-learning-databases/00501/PRSA2017_Data_20130301-20170228.zip',
    'https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip',
]

csv_path = DATA / f'{STATION.lower()}_air_quality.csv'
if not csv_path.exists():
    last_err = None
    for url in UCI_URLS:
        try:
            print('download', url)
            data = urlopen(url, timeout=120).read()
            with zipfile.ZipFile(BytesIO(data)) as zf:
                names = [n for n in zf.namelist() if STATION in n and n.endswith('.csv')]
                if not names:
                    raise FileNotFoundError(f'{STATION} não encontrado no zip')
                with zf.open(names[0]) as f:
                    df_raw = pd.read_csv(f)
            df_raw.to_csv(csv_path, index=False)
            print('salvo', csv_path)
            break
        except Exception as e:
            last_err = e
            print('falhou:', e)
    else:
        raise RuntimeError(f'não foi possível baixar dados: {last_err}')
else:
    df_raw = pd.read_csv(csv_path)
    print('já existe', csv_path, 'rows=', len(df_raw))

df_raw['timestamp'] = pd.to_datetime(df_raw[['year', 'month', 'day', 'hour']])
df_raw = df_raw.sort_values('timestamp').reset_index(drop=True)
print(df_raw[['timestamp', 'PM2.5', 'TEMP']].head(3))


## 4. Features Farooq + alvos AQI / P90


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

np.random.seed(PRIMARY_SEED)
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

EPA_BREAKPOINTS = [
    (12.0, 'Good'),
    (35.4, 'Moderate'),
    (55.4, 'Unhealthy_Sensitive'),
    (150.4, 'Unhealthy'),
    (250.4, 'Very_Unhealthy'),
    (float('inf'), 'Hazardous'),
]


def pm25_to_aqi_band(series: pd.Series) -> pd.Series:
    out = pd.Series(index=series.index, dtype=object)
    out[:] = 'Hazardous'
    prev = -float('inf')
    for hi, name in EPA_BREAKPOINTS:
        mask = (series > prev) & (series <= hi)
        out.loc[mask] = name
        prev = hi
    out.loc[series.isna()] = pd.NA
    return out


df = df_raw.copy()
for c in ['PM2.5', 'TEMP']:
    df[c] = df[c].ffill(limit=3)

source_map = {'PM2.5': 'pm25', 'TEMP': 'temperature'}
feat_cols = []
for src, prefix in source_map.items():
    roll = df[src].rolling(window=ROLL_WINDOW, min_periods=ROLL_MIN_PERIODS)
    df[f'{prefix}_min'] = roll.min()
    df[f'{prefix}_max'] = roll.max()
    df[f'{prefix}_median'] = roll.median()
    df[f'{prefix}_variance'] = roll.var()
    feat_cols += [f'{prefix}_min', f'{prefix}_max', f'{prefix}_median', f'{prefix}_variance']

df['future_pm25'] = df['PM2.5'].shift(-HORIZON_HOURS)
df = df.dropna(subset=['future_pm25'] + feat_cols).reset_index(drop=True)

n = len(df)
i_tr = int(n * 0.60)
i_va = i_tr + int(n * 0.20)
train_full = df.iloc[:i_tr].copy()
val_full = df.iloc[i_tr:i_va].copy()
test_full = df.iloc[i_va:].copy()

medians = train_full[feat_cols].median()
for part in (train_full, val_full, test_full):
    part[feat_cols] = part[feat_cols].fillna(medians)

# Extremo P90 (treino)
threshold = float(train_full['future_pm25'].quantile(EXTREME_PERCENTILE))
for part in (train_full, val_full, test_full):
    part['extreme_p90'] = (part['future_pm25'] >= threshold).astype(int)
    part['aqi_band'] = pm25_to_aqi_band(part['future_pm25'])
    part['aqi_bad'] = (~part['aqi_band'].isin(['Good', 'Moderate'])).astype(int)
    part['aqi_good_vs_moderate'] = np.where(
        part['aqi_band'] == 'Good', 0,
        np.where(part['aqi_band'] == 'Moderate', 1, np.nan),
    )

print('features:', feat_cols)
print('P90 threshold:', threshold)
print('sizes:', len(train_full), len(val_full), len(test_full))
print('faixas (treino):')
print(train_full['aqi_band'].value_counts().to_string())

# Distribuição das faixas
order = ['Good', 'Moderate', 'Unhealthy_Sensitive', 'Unhealthy', 'Very_Unhealthy', 'Hazardous']
counts = train_full['aqi_band'].value_counts().reindex([c for c in order if c in train_full['aqi_band'].unique()])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(counts.index.astype(str), counts.values, color='#4C78A8')
ax.set_title('Faixas AQI (EPA) no treino — future PM2.5 t+24h')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(PLOTS / 'aqi_band_distribution_train.png', dpi=140)
plt.show()


def stratified_subsample(part, size, seed, target_col='target'):
    size = min(size, len(part))
    if size < len(part):
        idx, _ = train_test_split(
            part.index, train_size=size, stratify=part[target_col], random_state=seed
        )
        out = part.loc[idx].sort_values('timestamp').copy()
    else:
        out = part.copy()
    return out.reset_index(drop=True)


export_cols = ['timestamp'] + feat_cols + [
    'PM2.5', 'TEMP', 'future_pm25', 'aqi_band', 'aqi_bad', 'aqi_good_vs_moderate', 'extreme_p90'
]
parts = []
for name, part in [('train', train_full), ('validation', val_full), ('test', test_full)]:
    tmp = part[[c for c in export_cols if c in part.columns]].copy()
    tmp['partition'] = name
    parts.append(tmp)
feat_path = WORK / 'features_with_aqi_bands.csv'
pd.concat(parts, ignore_index=True).to_csv(feat_path, index=False)
print('Salvo', feat_path)


## 5. Funções de treino (clássico 8D + QSVM PCA-2)


In [ ]:
import time
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score,
)
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel

try:
    from qiskit_machine_learning.state_fidelities import ComputeUncompute
    _HAS_CU = True
except Exception:
    _HAS_CU = False


def proba_pos(model, X):
    if hasattr(model, 'predict_proba'):
        p = model.predict_proba(X)
        if hasattr(model, 'classes_') and 1 in list(model.classes_):
            return p[:, list(model.classes_).index(1)]
        return p[:, -1]
    if hasattr(model, 'decision_function'):
        s = np.asarray(model.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-s))
    return None


def metrics_row(y_true, y_hat, y_prob, name, **extra):
    row = {
        'model': name,
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'average_precision': float(average_precision_score(y_true, y_prob)) if y_prob is not None else np.nan,
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_hat)),
        'precision_extreme': float(precision_score(y_true, y_hat, zero_division=0)),
        'recall_extreme': float(recall_score(y_true, y_hat, zero_division=0)),
        'f1_extreme': float(f1_score(y_true, y_hat, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_hat)),
        'auroc': float(roc_auc_score(y_true, y_prob)) if y_prob is not None and len(np.unique(y_true)) > 1 else np.nan,
    }
    row.update(extra)
    return row


def make_kernel(n_qubits, reps=1):
    fmap = ZZFeatureMap(feature_dimension=n_qubits, reps=reps, entanglement='linear')
    if _HAS_CU:
        fidelity = ComputeUncompute(sampler=StatevectorSampler())
        return FidelityQuantumKernel(feature_map=fmap, fidelity=fidelity, enforce_psd=True)
    return FidelityQuantumKernel(feature_map=fmap, enforce_psd=True)


def prepare_task_frames(target_col: str):
    """Copia partições e define coluna target a partir do alvo da tarefa."""
    frames = []
    for part in (train_full, val_full, test_full):
        p = part.dropna(subset=[target_col]).copy()
        p['target'] = p[target_col].astype(int)
        frames.append(p)
    return frames


def run_one_seed_task(train_df, test_df, seed, task_id, store_predictions=False):
    tr_s = stratified_subsample(train_df, TRAIN_SIZE, seed, target_col='target')
    te_s = stratified_subsample(test_df, TEST_SIZE, seed, target_col='target')

    if tr_s['target'].nunique() < 2 or te_s['target'].nunique() < 2:
        print(f'  SKIP seed={seed} task={task_id}: classes insuficientes')
        return [], {}, []

    pipe8 = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    X_tr8 = pipe8.fit_transform(tr_s[feat_cols])
    X_te8 = pipe8.transform(te_s[feat_cols])
    y_tr = tr_s['target'].to_numpy()
    y_te = te_s['target'].to_numpy()

    classicals = {
        'dummy': DummyClassifier(strategy='prior', random_state=seed),
        'logistic': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=seed),
        'svm_linear': CalibratedClassifierCV(
            SVC(kernel='linear', class_weight='balanced', random_state=seed), method='sigmoid', cv=3
        ),
        'svm_rbf': CalibratedClassifierCV(
            SVC(kernel='rbf', class_weight='balanced', random_state=seed), method='sigmoid', cv=3
        ),
    }

    rows, preds, pred_frames = [], {}, []

    for name, model in classicals.items():
        t0 = time.perf_counter()
        model.fit(X_tr8, y_tr)
        train_s = time.perf_counter() - t0
        y_hat = model.predict(X_te8)
        y_prob = proba_pos(model, X_te8)
        row = metrics_row(
            y_te, y_hat, y_prob, name,
            family='classical', n_features=8, seed=seed, task=task_id,
            training_seconds=train_s, positives_test=int(y_te.sum()),
        )
        rows.append(row)
        if store_predictions:
            preds[name] = {'y_true': y_te, 'y_hat': y_hat, 'y_prob': y_prob}
            pred_frames.append(pd.DataFrame({
                'task': task_id, 'model': name, 'seed': seed,
                'y_true': y_te, 'y_hat': y_hat, 'y_prob': y_prob,
            }))

    # QSVM PCA-2
    pipe_q = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=PCA_QUBITS, random_state=seed)),
    ])
    X_tr_q = pipe_q.fit_transform(tr_s[feat_cols])
    X_te_q = pipe_q.transform(te_s[feat_cols])

    for qid, ang_name, reps in Q_VARIANTS:
        if ang_name == 'minmax_0_1':
            ang = MinMaxScaler(feature_range=(0.0, 1.0))
        else:
            ang = MinMaxScaler(feature_range=(0.0, np.pi))
        X_tr_a = ang.fit_transform(X_tr_q)
        X_te_a = ang.transform(X_te_q)
        kernel = make_kernel(PCA_QUBITS, reps=reps)
        t0 = time.perf_counter()
        K_tr = kernel.evaluate(x_vec=X_tr_a)
        K_te = kernel.evaluate(x_vec=X_te_a, y_vec=X_tr_a)
        ks = time.perf_counter() - t0
        clf = SVC(kernel='precomputed', class_weight='balanced', random_state=seed)
        clf.fit(K_tr, y_tr)
        y_hat = clf.predict(K_te)
        y_prob = proba_pos(clf, K_te)
        row = metrics_row(
            y_te, y_hat, y_prob, qid,
            family='qsvm', n_features=2, seed=seed, task=task_id,
            angular_scaler=ang_name, reps=reps, kernel_seconds=ks,
            positives_test=int(y_te.sum()),
        )
        rows.append(row)
        if store_predictions:
            preds[qid] = {'y_true': y_te, 'y_hat': y_hat, 'y_prob': y_prob}
            pred_frames.append(pd.DataFrame({
                'task': task_id, 'model': qid, 'seed': seed,
                'y_true': y_te, 'y_hat': y_hat, 'y_prob': y_prob,
            }))
        print(f'  [{task_id}] {qid}: AUPRC={row["average_precision"]:.3f} F1={row["f1_extreme"]:.3f} ({ks:.1f}s)')

    return rows, preds, pred_frames

print('funções ok')


## 6. Loop multi-tarefa × multi-seed


In [ ]:
from IPython.display import display
from scipy.stats import wilcoxon
import json

all_rows = []
all_pred_frames = []
predictions_primary = {}  # task -> model -> preds (PRIMARY_SEED)
t_global = time.perf_counter()

for task_id, target_col, task_label in TASKS:
    print('\n' + '=' * 70)
    print(f'TASK {task_id} — {task_label}')
    print('=' * 70)
    tr_df, va_df, te_df = prepare_task_frames(target_col)
    print('pos rate train/test:', float(tr_df['target'].mean()), float(te_df['target'].mean()),
          'n=', len(tr_df), len(te_df))

    for i, seed in enumerate(SEEDS, 1):
        store = seed == PRIMARY_SEED
        print(f'[{i}/{len(SEEDS)}] seed={seed}')
        rows, preds, pred_frames = run_one_seed_task(tr_df, te_df, seed, task_id, store_predictions=store)
        all_rows.extend(rows)
        all_pred_frames.extend(pred_frames)
        if store and preds:
            predictions_primary[task_id] = preds

runs_raw = pd.DataFrame(all_rows)
runs_raw.to_csv(WORK / 'runs_raw.csv', index=False)

summary = (
    runs_raw.groupby(['task', 'model', 'family'], as_index=False)
    .agg(
        n_seeds=('seed', 'nunique'),
        average_precision_mean=('average_precision', 'mean'),
        average_precision_std=('average_precision', 'std'),
        f1_mean=('f1_extreme', 'mean'),
        recall_mean=('recall_extreme', 'mean'),
        auroc_mean=('auroc', 'mean'),
        accuracy_mean=('accuracy', 'mean'),
    )
)
summary.to_csv(WORK / 'summary_mean_std.csv', index=False)

# Melhor modelo por tarefa (AUPRC médio) — útil para a Névoa
best_by_task = (
    summary.sort_values(['task', 'average_precision_mean'], ascending=[True, False])
    .groupby('task', as_index=False)
    .first()
)
best_by_task.to_csv(WORK / 'best_model_by_task.csv', index=False)
print('\n=== melhor modelo por tarefa ===')
display(best_by_task[['task', 'model', 'family', 'average_precision_mean', 'f1_mean']])

print('\n=== summary ===')
display(summary.sort_values(['task', 'average_precision_mean'], ascending=[True, False]))

# Wilcoxon QSVM vs melhor clássico, por tarefa (se >= 2 seeds)
wilcoxon_rows = []
for task_id, _, _ in TASKS:
    sub = runs_raw[runs_raw['task'] == task_id]
    if sub['seed'].nunique() < 2:
        continue
    q = sub[sub['family'] == 'qsvm']
    if q.empty:
        continue
    best_q = q.groupby('model')['average_precision'].mean().idxmax()
    c = sub[sub['family'] == 'classical']
    best_c = c.groupby('model')['average_precision'].mean().idxmax()
    q_scores = q[q['model'] == best_q].set_index('seed')['average_precision']
    c_scores = c[c['model'] == best_c].set_index('seed')['average_precision']
    common = q_scores.index.intersection(c_scores.index)
    if len(common) < 2:
        continue
    diff = (q_scores.loc[common] - c_scores.loc[common]).to_numpy()
    try:
        stat, pval = wilcoxon(diff)
    except ValueError:
        stat, pval = np.nan, np.nan
    wilcoxon_rows.append({
        'task': task_id, 'qsvm': best_q, 'classical': best_c,
        'delta_mean': float(np.mean(diff)), 'wilcoxon_stat': float(stat) if stat == stat else None,
        'pvalue': float(pval) if pval == pval else None, 'n_seeds': len(common),
    })
if wilcoxon_rows:
    wdf = pd.DataFrame(wilcoxon_rows)
    wdf.to_csv(WORK / 'wilcoxon_by_task.csv', index=False)
    display(wdf)

if all_pred_frames:
    pd.concat(all_pred_frames, ignore_index=True).to_csv(WORK / 'predictions_primary_seed.csv', index=False)

print(f'\nTempo total: {(time.perf_counter() - t_global)/60:.1f} min')


## 7. Painel visual por tarefa


In [ ]:
import seaborn as sns
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay, confusion_matrix

# Barras AUPRC por tarefa
tasks_present = summary['task'].unique().tolist()
fig, axes = plt.subplots(1, len(tasks_present), figsize=(5.2 * len(tasks_present), 4.2), sharey=False)
if len(tasks_present) == 1:
    axes = [axes]
for ax, task_id in zip(axes, tasks_present):
    sub = summary[summary['task'] == task_id].sort_values('average_precision_mean')
    colors = ['#54A24B' if f == 'qsvm' else '#4C78A8' for f in sub['family']]
    ax.barh(sub['model'], sub['average_precision_mean'], xerr=sub['average_precision_std'].fillna(0),
            color=colors, capsize=3)
    ax.set_xlabel('AUPRC')
    ax.set_title(task_id)
fig.suptitle('Legado Farooq — AUPRC por tarefa (média ± std)', y=1.02)
fig.tight_layout()
fig.savefig(PLOTS / 'auprc_by_task.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusão + PR/ROC para seed primária, uma figura por tarefa
for task_id, preds in predictions_primary.items():
    names = list(preds.keys())
    ncols = 3
    nrows = int(np.ceil(len(names) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, name in zip(axes, names):
        p = preds[name]
        cm = confusion_matrix(p['y_true'], p['y_hat'], labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
        ax.set_title(name)
        ax.set_xlabel('pred'); ax.set_ylabel('true')
    for ax in axes[len(names):]:
        ax.axis('off')
    fig.suptitle(f'Confusão — {task_id} (seed {PRIMARY_SEED})', y=1.02)
    fig.tight_layout()
    fig.savefig(PLOTS / f'confusion_{task_id}.png', dpi=140, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for name, p in preds.items():
        if p['y_prob'] is None or len(np.unique(p['y_true'])) < 2:
            continue
        PrecisionRecallDisplay.from_predictions(p['y_true'], p['y_prob'], name=name, ax=axes[0])
        RocCurveDisplay.from_predictions(p['y_true'], p['y_prob'], name=name, ax=axes[1])
    axes[0].set_title(f'PR — {task_id}'); axes[1].set_title(f'ROC — {task_id}')
    axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(PLOTS / f'pr_roc_{task_id}.png', dpi=140, bbox_inches='tight')
    plt.show()


## 8. Export para a Névoa


In [ ]:
from IPython.display import display
# Mapeia tarefa -> modelo preferido (nomes da UI)
UI_MODEL = {
    'dummy': 'logreg',
    'logistic': 'logreg',
    'svm_linear': 'svm_rbf',
    'svm_rbf': 'svm_rbf',
    'Q_farooq_pipeline': 'qsvm',
    'Q_farooq_0pi': 'qsvm',
}

# Sugestão por faixa da Névoa
# Good/Moderate <- aqi_good_vs_moderate
# Unhealthy_* / Hazardous <- aqi_bad (e extreme_p90 como reforço)
recs = []
for _, row in best_by_task.iterrows():
    recs.append({
        'task': row['task'],
        'best_model_raw': row['model'],
        'preferredModel': UI_MODEL.get(row['model'], 'qsvm'),
        'average_precision_mean': row['average_precision_mean'],
        'f1_mean': row['f1_mean'],
        'family': row['family'],
    })

ui_map = {
    'Good': 'aqi_good_vs_moderate',
    'Moderate': 'aqi_good_vs_moderate',
    'Unhealthy_Sensitive': 'aqi_bad',
    'Unhealthy': 'aqi_bad',
    'Very_Unhealthy': 'extreme_p90',
    'Hazardous': 'extreme_p90',
}
task_best = {r['task']: r for r in recs}
band_recs = []
for band, task in ui_map.items():
    r = task_best.get(task)
    if not r:
        continue
    band_recs.append({
        'band': band,
        'task': task,
        'preferredModel': r['preferredModel'],
        'modelWhy': (
            f"Melhor AUPRC no legado Farooq multi-seed ({MODE}): "
            f"{r['best_model_raw']} = {r['average_precision_mean']:.3f}"
        ),
    })

meta = {
    'mode': MODE,
    'seeds': SEEDS,
    'station': STATION,
    'horizon_hours': HORIZON_HOURS,
    'p90_threshold': threshold,
    'tasks': [t[0] for t in TASKS],
    'features': feat_cols,
    'best_by_task': recs,
    'band_recommendations': band_recs,
    'note': 'Legado Farooq adaptado para faixas AQI (t+24). Não é o fair benchmark.',
}
(WORK / 'meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
pd.DataFrame(band_recs).to_csv(WORK / 'nevoa_band_recommendations.csv', index=False)
print('Exportado:')
print(' ', WORK / 'meta.json')
print(' ', WORK / 'nevoa_band_recommendations.csv')
print(' ', WORK / 'best_model_by_task.csv')
print(' ', WORK / 'summary_mean_std.csv')
display(pd.DataFrame(band_recs))


### Como usar

1. Escolha `MODE = 'small' | 'medium' | 'large'`
2. Run All
3. Artefatos em `artifacts/kaggle_farooq_aqi_nb/` (ou `/kaggle/working`)
4. Use `nevoa_band_recommendations.csv` / `best_model_by_task.csv` para atualizar preferências da UI

Tarefas:
- **aqi_good_vs_moderate** → faixas Bom/Moderado (próximo ao artigo)
- **aqi_bad** → alerta “pior que Moderado”
- **extreme_p90** → episódio extremo
